# NB_SETUP_DEVELOPER_WORKSPACES

Maakt en beheert de developer-workspaces van één developer (ADR-011): `DEV_<NAAM>_<prefix>_{DATA,CODE,SEMANTIC,REPORTING}` voor een business domain, en `DEV_<NAAM>_<prefix>_INTEGRATION_{DATA,CODE}`. Elke workspace hangt aan een feature-branch in de repo van zijn Dev-workspace.

| mode | Wat | Wanneer |
|---|---|---|
| `create` | Workspaces aanmaken, rechten overnemen van de Dev-workspace, koppelen aan `branch`, INTEGRATION DATA registreren in de configdb, daarna `bind` | Eenmalig per developer |
| `new_feature` | Dezelfde workspaces op een nieuwe `branch` vanaf `dev` zetten, daarna `bind` | Nieuwe feature in hetzelfde domein |
| `switch_domain` | De domein-workspaces **leeggooien** en koppelen aan `domain`; INTEGRATION alleen een nieuwe branch; daarna `bind` | Ander business domain |
| `bind` | Defaults van de libraries, model en rapport naar de eigen workspaces; controle op `%%configure` en vangnet | Na elke *Update from Git* in je developer-workspace |

Voorwaarden: `create`, `new_feature` en `switch_domain` stoppen als een workspace niet-gecommitte wijzigingen heeft (commit ook als het alleen bindingen zijn). Wie dit draait, is Admin op de developer-workspaces en minstens Contributor op de Dev-workspaces. De GitHub-token (bootstrap-cel) mag branches aanmaken in de repo's per workspace.

Alleen een wijziging in model en rapport? Zet `workspaces` op `["SEMANTIC", "REPORTING"]`. Het model blijft dan op Gold (D) en alleen het rapport gaat naar je eigen model.

Werkinstructie: *Ontwikkelproces* (stap 1 tot en met 8).

# Bootstrap en manifest

In [ ]:
# =============================================================================
# BOOTSTRAP - waar staat je fork?
# Deze vier regels worden gestempeld door NB_BOOTSTRAP_FMD. Pas ze alleen met de
# hand aan als je dit notebook los importeert.
# =============================================================================
repo_owner    = "bitmetric-fabric"
repo_name     = "FMD_FRAMEWORK"
branch        = "main"
folder_prefix = ""
github_token_key_vault = ""
github_token_secret    = ""

# =============================================================================
# Manifest (ADR-003)
# Alles wat per klant vaststaat en daarna niet meer verandert, staat in
# manifest.yaml in de root van de repo hierboven. Niets daarvan hoort nog in een
# notebookcel. Zie manifest.example.yaml voor de volledige structuur.
# =============================================================================
import requests as _requests
import yaml as _yaml

MANIFEST_URL = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/{branch}/manifest.yaml"

github_token = (
    notebookutils.credentials.getSecret(
        f"https://{github_token_key_vault}.vault.azure.net/", github_token_secret
    )
    if github_token_key_vault and github_token_secret
    else None
)
github_headers = {"Authorization": f"Bearer {github_token}"} if github_token else {}

_response = _requests.get(MANIFEST_URL, headers=github_headers)
if _response.status_code == 404:
    raise RuntimeError(
        f"Geen manifest.yaml gevonden op {MANIFEST_URL}\n"
        f"Kopieer manifest.example.yaml naar manifest.yaml in de root van "
        f"{repo_owner}/{repo_name} (branch '{branch}'), vul hem in en commit hem."
    )
_response.raise_for_status()
manifest = _yaml.safe_load(_response.text) or {}


def _lookup(path):
    """Geneste opzoeking op 'a.b.c'. Geeft (waarde, gevonden) terug."""
    node = manifest
    for part in path.split("."):
        if not isinstance(node, dict) or part not in node:
            return None, False
        node = node[part]
    return node, True


# Alles wat dit notebook nodig heeft. In een keer gecontroleerd, zodat je niet
# per ontbrekende sleutel opnieuw hoeft te draaien.
REQUIRED = [
    "repository.owner",
    "repository.name",
    "repository.branch",
    "naming.domain_name",
    "naming.framework_pre_fix",
    "naming.business_domains",
    "environments",
    "developer_workspaces.prefix",
    "developer_workspaces.capacity",
    "developer_workspaces.git.owner",
    "developer_workspaces.git.connection_id",
    "developer_workspaces.git.base_branch",
    "developer_workspaces.git.repository",
]

_missing = [path for path in REQUIRED if not _lookup(path)[1]]
if _missing:
    raise RuntimeError(
        "manifest.yaml mist verplichte sleutel(s):\n"
        + "\n".join(f"  - {path}" for path in _missing)
        + f"\n\nBron: {MANIFEST_URL}"
        + "\nZie manifest.example.yaml voor de volledige structuur."
    )


def require(path):
    """Verplichte manifest-waarde."""
    value, found = _lookup(path)
    if not found:
        raise KeyError(f"'{path}' staat niet in manifest.yaml ({MANIFEST_URL})")
    return value


def optional(path, default=None):
    """Manifest-waarde die weg mag zijn."""
    value, found = _lookup(path)
    return value if found else default


def ws_suffix(short):
    """Omgevingsmarker voor workspacenamen. Production krijgt er geen: alleen
    Dev/Test/Acceptance worden gemarkeerd, bv. 'INTEGRATION DATA (D)' naast
    'INTEGRATION DATA' in productie."""
    return "" if short == "P" else f" ({short})"


# De bootstrap-cel en het manifest moeten dezelfde repo aanwijzen; anders komen
# src/ en config/ straks ergens anders vandaan dan het manifest beschrijft.
for _key, _bootstrap_value in (
    ("repository.owner", repo_owner),
    ("repository.name", repo_name),
    ("repository.branch", branch),
):
    if str(require(_key)) != str(_bootstrap_value):
        raise RuntimeError(
            f"Bootstrap en manifest spreken elkaar tegen: {_key} is "
            f"'{require(_key)}' in manifest.yaml maar '{_bootstrap_value}' in de "
            f"bootstrap-cel hierboven."
        )

print(f"Manifest geladen van {MANIFEST_URL}")


# Parameters

In [ ]:
# =============================================================================
# Per run invullen
# =============================================================================
developer     = ""            # bijvoorbeeld "MITCHELL": alleen hoofdletters en cijfers
mode          = "bind"        # create | new_feature | switch_domain | bind
domain        = ""            # business domain voor DATA/CODE/SEMANTIC/REPORTING, bijvoorbeeld "FINANCE"
branch        = ""            # bijvoorbeeld "feature/valutanamen"; niet nodig voor bind
workspaces    = ["DATA", "CODE", "SEMANTIC", "REPORTING", "INTEGRATION_DATA", "INTEGRATION_CODE"]
confirm_empty = False         # switch_domain gooit je domein-workspaces leeg, ook je eigen Gold-data: True om te bevestigen

# Functies

In [ ]:
# =============================================================================
# Functies. Gebaseerd op klant/tools/dev_bind.py en switch.py uit FMD_TRIAL
# (testplan-devflow F2 tot en met F9, 2026-09-29).
# =============================================================================
import base64
import json
import re
import struct
import time

import pyodbc
import requests

API = "https://api.fabric.microsoft.com/v1"
PBI = "https://api.powerbi.com/v1.0/myorg"
GITHUB = "https://api.github.com"
DEFAULT_SET = "Default value set"
DOMAIN_TYPES = ["DATA", "CODE", "SEMANTIC", "REPORTING"]
INTEGRATION_TYPES = ["INTEGRATION_DATA", "INTEGRATION_CODE"]
SQL_DB = re.compile(r'Sql\.Database\("([^"]+)",\s*"([^"]+)"\)')

pre_fix = require("naming.framework_pre_fix")
integration_domain = require("naming.domain_name")
dev_short = require("environments")[0]["short"]
dw = require("developer_workspaces")
git_owner = dw["git"]["owner"]
git_connection_id = dw["git"]["connection_id"]
base_branch = dw["git"]["base_branch"]
repo_pattern = dw["git"]["repository"]


def headers(audience="pbi"):
    # Het token verloopt na ongeveer een uur; daarom per aanroep opnieuw ophalen.
    return {"Authorization": f"Bearer {notebookutils.credentials.getToken(audience)}", "Content-Type": "application/json"}


def call(method, url, body=None, ok_statuses=(200, 201, 202), audience="pbi"):
    """Fabric- of Power BI-aanroep. Wacht op een long-running operation (202 met Location)."""
    url = url if url.startswith("http") else f"{API}/{url}"
    r = requests.request(method, url, headers=headers(audience), json=body)
    if r.status_code == 202 and "Location" in r.headers:
        loc = r.headers["Location"]
        while True:
            time.sleep(int(r.headers.get("Retry-After", 5)))
            r = requests.get(loc, headers=headers(audience))
            state = r.json().get("status")
            if state == "Failed":
                raise RuntimeError(f"{method} {url}: {r.text}")
            if state == "Succeeded":
                res = requests.get(loc + "/result", headers=headers(audience))
                return res.json() if res.ok and res.text else {}
    if r.status_code not in ok_statuses:
        raise RuntimeError(f"{method} {url}: {r.status_code} {r.text}")
    return r.json() if r.text else {}


def github(method, path, body=None):
    if not github_token:
        raise RuntimeError("Geen GitHub-token: vul github_token_key_vault en github_token_secret in de bootstrap-cel.")
    r = requests.request(method, f"{GITHUB}/{path}", json=body,
                         headers={"Authorization": f"Bearer {github_token}", "Accept": "application/vnd.github+json"})
    return r


def workspace_name(kind):
    """Naam van de Dev-workspace (Base) voor 'DATA' of 'INTEGRATION_DATA'."""
    group, ws_type = (integration_domain, kind.split("_", 1)[1]) if kind.startswith("INTEGRATION_") else (domain, kind)
    return f"{pre_fix + ' ' if pre_fix else ''}{group} {ws_type}{ws_suffix(dev_short)}"


def dev_workspace_name(kind):
    return f"DEV_{developer}_{dw['prefix']}_{kind}"


def repo_name(kind):
    group, ws_type = (integration_domain, kind.split("_", 1)[1]) if kind.startswith("INTEGRATION_") else (domain, kind)
    return repo_pattern.format(pre_fix=pre_fix, domain=group, type=ws_type)


_workspaces = None


def find_workspace(name):
    global _workspaces
    if _workspaces is None:
        _workspaces = {w["displayName"]: w["id"] for w in call("GET", "workspaces")["value"]}
    return _workspaces.get(name)


def items(ws, item_type=None):
    return call("GET", f"workspaces/{ws}/items" + (f"?type={item_type}" if item_type else ""))["value"]


def git_status(ws):
    return call("GET", f"workspaces/{ws}/git/status")


def git_repo_of(ws):
    details = call("GET", f"workspaces/{ws}/git/connection").get("gitProviderDetails") or {}
    return details.get("repositoryName"), details.get("branchName")


def domain_of(ws):
    """Het domein van een workspace: de reponaam zonder het type (fmd_trial_finance_data -> fmd_trial_finance)."""
    repo, _ = git_repo_of(ws)
    return re.sub(r"_(data|code|semantic|reporting)$", "", repo.lower()) if repo else None


def require_clean(kinds):
    """Stopt als een workspace niet-gecommitte of niet-gesynchroniseerde wijzigingen heeft."""
    for kind in kinds:
        ws = find_workspace(dev_workspace_name(kind))
        if not ws or git_repo_of(ws)[0] is None:
            continue
        s = git_status(ws)
        if s.get("changes") or s.get("workspaceHead") != s.get("remoteCommitHash"):
            raise RuntimeError(f"{dev_workspace_name(kind)} heeft niet-gecommitte of niet-gesynchroniseerde wijzigingen. "
                               "Commit eerst (ook als het alleen bindingen zijn); er is niets gewijzigd.")


def ensure_branch(repo):
    r = github("GET", f"repos/{git_owner}/{repo}/branches/{branch}")
    if r.status_code == 200:
        return "bestond al"
    base = github("GET", f"repos/{git_owner}/{repo}/git/ref/heads/{base_branch}")
    if base.status_code in (401, 403, 404):
        raise RuntimeError(f"GitHub {base.status_code} op {git_owner}/{repo}@{base_branch}: bestaat de repo en de branch, en heeft "
                           "de GitHub-token uit de bootstrap-cel toegang (Contents: read and write)?")
    base.raise_for_status()
    sha = base.json()["object"]["sha"]
    github("POST", f"repos/{git_owner}/{repo}/git/refs", {"ref": f"refs/heads/{branch}", "sha": sha}).raise_for_status()
    return f"aangemaakt vanaf {base_branch} ({sha[:8]})"


def empty_workspace(ws, name):
    """Domeinwissel: alles weg, zodat items met dezelfde naam maar een andere logical ID niet botsen (F8)."""
    if git_repo_of(ws)[0]:
        call("POST", f"workspaces/{ws}/git/disconnect")
    # eerst wat verwijst: een notebook waar een pipeline naar verwijst, geeft bij DELETE een UnknownError
    order = ["Report", "SemanticModel", "DataPipeline", "Notebook", "VariableLibrary", "Lakehouse"]
    for item in sorted(items(ws), key=lambda i: order.index(i["type"]) if i["type"] in order else 2.5):
        if item["type"] != "SQLEndpoint":  # gaat mee met het lakehouse
            call("DELETE", f"workspaces/{ws}/items/{item['id']}")
            print(f"  {name}: {item['displayName']} ({item['type']}) verwijderd")


def connect(ws, name, kind):
    """Koppelt aan <repo>@<branch>, haalt de inhoud binnen en legt de relatie Base."""
    repo = repo_name(kind)
    print(f"  {name} -> {repo}@{branch}: branch {ensure_branch(repo)}")
    if git_repo_of(ws)[0]:
        call("POST", f"workspaces/{ws}/git/disconnect")
    call("POST", f"workspaces/{ws}/git/connect", {
        "gitProviderDetails": {"gitProviderType": "GitHub", "ownerName": git_owner, "repositoryName": repo,
                               "branchName": branch, "directoryName": "/"},
        "myGitCredentials": {"source": "ConfiguredConnection", "connectionId": git_connection_id}})
    init = call("POST", f"workspaces/{ws}/git/initializeConnection", {"initializationStrategy": "PreferRemote"})
    if init.get("requiredAction") == "UpdateFromGit":
        body = {"remoteCommitHash": init["remoteCommitHash"], "options": {"allowOverrideItems": True},
                "conflictResolution": {"conflictResolutionType": "Workspace", "conflictResolutionPolicy": "PreferRemote"}}
        if init.get("workspaceHead"):  # een lege workspace heeft geen head; null geeft UnknownError (F8)
            body["workspaceHead"] = init["workspaceHead"]
        for attempt in range(6):
            try:
                call("POST", f"workspaces/{ws}/git/updateFromGit", body)
                break
            except RuntimeError as e:
                # net na leeggooien houdt Fabric de naam van een verwijderd item nog even vast: UnknownError
                if "UnknownError" not in str(e) or attempt == 5:
                    raise
                print(f"  {name}: update mislukt (naam nog bezet na verwijderen?), opnieuw over 60 s")
                time.sleep(60)
    base = find_workspace(workspace_name(kind))
    relations = call("GET", f"workspaces/{ws}/git/workspaceRelations").get("value", [])
    if base and not any(r["relationType"] == "Base" and r["relatedWorkspaceId"] == base for r in relations):
        try:  # preview; alleen voor de weergave in de portal
            call("POST", f"workspaces/{ws}/git/workspaceRelations", {"relatedWorkspaceId": base, "relationType": "Base"})
        except RuntimeError as e:
            print(f"  {name}: relatie Base niet gelegd ({str(e)[:120]})")
    s = git_status(ws)
    print(f"  {name}: {len(items(ws))} items, {len(s.get('changes', []))} wijzigingen")


def copy_roles(ws, name, kind):
    """Branch out en een nieuwe workspace nemen de rechten niet over. Groepen en service principals van de
    Dev-workspace krijgen dezelfde rol; gebruikers en workspace-identities niet."""
    base = find_workspace(workspace_name(kind))
    if not base:
        print(f"  {name}: geen {workspace_name(kind)}; rechten overgeslagen")
        return
    have = {a["principal"]["id"] for a in call("GET", f"workspaces/{ws}/roleAssignments")["value"]}
    for a in call("GET", f"workspaces/{base}/roleAssignments")["value"]:
        p = a["principal"]
        if p["type"] not in ("Group", "ServicePrincipal") or p["id"] in have or p.get("displayName") in (_workspaces or {}):
            continue
        call("POST", f"workspaces/{ws}/roleAssignments", {"principal": {"id": p["id"], "type": p["type"]}, "role": a["role"]})
        print(f"  {name}: {p.get('displayName')} toegevoegd als {a['role']}")


def lakehouse(ws, name):
    return next((l for l in call("GET", f"workspaces/{ws}/lakehouses")["value"] if l["displayName"] == name), None)


def gold_built(ws, lh):
    r = requests.get(f"https://onelake.dfs.fabric.microsoft.com/{ws}", headers=headers("storage"),
                     params={"resource": "filesystem", "recursive": "false", "directory": f"{lh}/Tables/gold"})
    return r.ok and any(not p["name"].endswith(".json.gz") for p in r.json().get("paths", []))


def library_files(ws, lib_name):
    lib = next((i for i in items(ws, "VariableLibrary") if i["displayName"] == lib_name), None)
    if not lib:
        return None, None
    parts = call("POST", f"workspaces/{ws}/items/{lib['id']}/getDefinition")["definition"]["parts"]
    return lib, {p["path"]: base64.b64decode(p["payload"]).decode("utf-8") for p in parts}


def bind_defaults(ws, lib_name, overrides):
    """Defaults op de eigen ID's, met de default actief. Geen value set per developer (F7)."""
    lib, files = library_files(ws, lib_name)
    if not lib:
        print(f"  {lib_name} niet gevonden; overgeslagen")
        return
    active = call("GET", f"workspaces/{ws}/variableLibraries/{lib['id']}")["properties"].get("activeValueSetName")
    if active != DEFAULT_SET:
        call("PATCH", f"workspaces/{ws}/variableLibraries/{lib['id']}", {"properties": {"activeValueSetName": DEFAULT_SET}})
    variables = json.loads(files["variables.json"])
    if all(v["value"] == overrides[v["name"]] for v in variables["variables"] if v["name"] in overrides):
        print(f"  {lib_name}: defaults stonden al op de eigen ID's")
        return
    for v in variables["variables"]:
        if v["name"] in overrides:
            v["value"] = overrides[v["name"]]
    files["variables.json"] = json.dumps(variables, indent=2)
    call("POST", f"workspaces/{ws}/items/{lib['id']}/updateDefinition", {"definition": {"parts": [
        {"path": k, "payload": base64.b64encode(v.encode("utf-8")).decode(), "payloadType": "InlineBase64"}
        for k, v in files.items() if k != ".platform"]}})
    print(f"  {lib_name}: defaults op de eigen ID's")


def bind_models(sem_ws, server, endpoint_id, rebind):
    models = {}
    for m in items(sem_ws, "SemanticModel"):
        models[m["displayName"]] = m["id"]
        parts = call("POST", f"workspaces/{sem_ws}/items/{m['id']}/getDefinition")["definition"]["parts"]
        changed = False
        for p in parts:
            if not p["path"].endswith(".tmdl"):
                continue
            src = base64.b64decode(p["payload"]).decode("utf-8")
            if not SQL_DB.search(src):
                continue
            if not rebind:
                print(f"  model {m['displayName']}: blijft op {SQL_DB.search(src).group(2)} (Gold (D))")
                continue
            new = SQL_DB.sub(f'Sql.Database("{server}", "{endpoint_id}")', src)
            if new != src:
                p["payload"] = base64.b64encode(new.encode("utf-8")).decode()
                changed = True
        if changed:
            call("POST", f"workspaces/{sem_ws}/items/{m['id']}/updateDefinition",
                 {"definition": {"parts": [p for p in parts if p["path"] != ".platform"]}})
            print(f"  model {m['displayName']}: gekoppeld aan de eigen Gold ({endpoint_id})")
    return models


def bind_reports(rep_ws, models):
    for r in call("GET", f"{PBI}/groups/{rep_ws}/reports")["value"]:
        if r["datasetId"] in models.values():
            print(f"  rapport {r['name']}: wijst al naar het eigen model")
            continue
        ds_name = call("GET", f"{PBI}/datasets/{r['datasetId']}")["name"]
        target = models.get(ds_name)
        if not target:
            print(f"  rapport {r['name']}: geen model {ds_name} in de eigen SEMANTIC; niet omgezet")
            continue
        call("POST", f"{PBI}/groups/{rep_ws}/reports/{r['id']}/Rebind", {"datasetId": target})
        print(f"  rapport {r['name']}: gekoppeld aan het eigen {ds_name}")


def check_notebooks(code_ws):
    """Stap 3 van de werkinstructie: elke notebook met een default lakehouse heeft %%configure en het vangnet."""
    for nb in items(code_ws, "Notebook"):
        parts = call("POST", f"workspaces/{code_ws}/items/{nb['id']}/getDefinition")["definition"]["parts"]
        src = next((base64.b64decode(p["payload"]).decode("utf-8") for p in parts if p["path"].startswith("notebook-content")), "")
        if re.search(r'"default_lakehouse":\s*"[0-9a-f-]{36}"', src) and ("%%configure" not in src or "Vangnet" not in src):
            print(f"  LET OP: {nb['displayName']} heeft een default lakehouse maar geen %%configure en/of vangnet; "
                  "hij kan in een gedeelde Gold schrijven.")


def register_integration_data(int_data):
    """De INTEGRATION DATA van de developer in de configdb, zodat PL_FMD_LOAD_ALL in de eigen lakehouses laadt."""
    base_code = find_workspace(workspace_name("INTEGRATION_CODE"))
    _, files = library_files(base_code, "VAR_CONFIG_FMD")
    config = {v["name"]: v["value"] for v in json.loads(files["variables.json"])["variables"]}
    token = notebookutils.credentials.getToken("pbi").encode("utf-16-le")
    connection = pyodbc.connect(f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={config['fmd_fabric_db_connection']};"
                                f"DATABASE={config['fmd_fabric_db_name']};",
                                attrs_before={1256: struct.pack(f"<I{len(token)}s", len(token), token)}, timeout=30)
    cursor = connection.cursor()
    name = dev_workspace_name("INTEGRATION_DATA")
    cursor.execute("EXEC [integration].[sp_UpsertWorkspace] @WorkspaceId = ?, @Name = ?", int_data, name)
    for lh_name in ("LH_DATA_LANDINGZONE", "LH_BRONZE_LAYER", "LH_SILVER_LAYER"):
        lh = lakehouse(int_data, lh_name)
        cursor.execute("EXEC [integration].[sp_UpsertLakehouse] @LakehouseId = ?, @WorkspaceId = ?, @Name = ?", lh["id"], int_data, lh_name)
        print(f"  configdb: {lh_name} ({lh['id']})")
    connection.commit()
    connection.close()


def bind():
    """Isoleert de developer-workspaces. Draai na elke Update from Git in je developer-workspace."""
    ws = {k: find_workspace(dev_workspace_name(k)) for k in DOMAIN_TYPES + INTEGRATION_TYPES}
    dom = {k: domain_of(w) if w else None for k, w in ws.items() if k in DOMAIN_TYPES}
    print(f"Domeinen: {dom}")
    gold = lakehouse(ws["DATA"], "LH_GOLD_LAYER") if ws["DATA"] else None
    built = bool(gold) and gold_built(ws["DATA"], gold["id"])
    print(f"Eigen Gold: {gold['id'] + (' (gebouwd)' if built else ' (niet gebouwd)') if gold else 'geen'}")

    print("Variable Libraries:")
    if ws["CODE"] and gold and dom["CODE"] == dom["DATA"]:
        bind_defaults(ws["CODE"], "VAR_GOLD_SHORTCUTS_FMD", {"SourceWorkspaceId": ws["DATA"], "SourceLakehouseId": gold["id"]})
    elif ws["CODE"]:
        print("  VAR_GOLD_SHORTCUTS_FMD overgeslagen: CODE en DATA horen niet bij hetzelfde domein")
    if ws["INTEGRATION_CODE"] and ws["INTEGRATION_DATA"]:
        ldz = lakehouse(ws["INTEGRATION_DATA"], "LH_DATA_LANDINGZONE")
        bind_defaults(ws["INTEGRATION_CODE"], "VAR_CONFIG_FMD",
                      {"fmd_data_workspace_guid": ws["INTEGRATION_DATA"], "fmd_landingzone_lakehouse_guid": ldz["id"]})

    print("Model en rapport:")
    rebind = bool(gold) and built and dom.get("SEMANTIC") == dom["DATA"]
    if ws["SEMANTIC"]:
        if not rebind:
            print("  model niet omgezet: DATA hoort bij een ander domein, of de eigen Gold is niet gebouwd; het model leest Gold (D)")
        sql = gold["properties"]["sqlEndpointProperties"] if gold else {}
        models = bind_models(ws["SEMANTIC"], sql.get("connectionString"), sql.get("id"), rebind)
        if ws["REPORTING"]:
            bind_reports(ws["REPORTING"], models)
    if ws["CODE"]:
        check_notebooks(ws["CODE"])
    print("Klaar. Ververs het model voordat je het rapport opent.")


def run():
    global _workspaces
    if not re.fullmatch(r"[A-Z0-9]+", developer):
        raise ValueError("developer: alleen hoofdletters en cijfers, bijvoorbeeld MITCHELL")
    if mode != "bind" and not branch:
        raise ValueError("branch is verplicht voor create, new_feature en switch_domain")
    kinds = [k for k in workspaces if k in DOMAIN_TYPES + INTEGRATION_TYPES]

    if mode == "create":
        capacity = next((c["id"] for c in call("GET", "capacities")["value"] if c["displayName"] == dw["capacity"]), None)
        if not capacity:
            raise RuntimeError(f"Capaciteit {dw['capacity']} niet gevonden")
        for kind in kinds:
            name = dev_workspace_name(kind)
            ws = find_workspace(name)
            repo, br = git_repo_of(ws) if ws else (None, None)
            if repo and (repo.lower(), br) != (repo_name(kind).lower(), branch):
                raise RuntimeError(f"{name} is gekoppeld aan {repo}@{br}: gebruik new_feature of switch_domain")
            if not ws:
                ws = call("POST", "workspaces", {"displayName": name, "capacityId": capacity})["id"]
                _workspaces[name] = ws
                print(f"{name}: aangemaakt ({ws})")
            copy_roles(ws, name, kind)
            if repo:  # een eerdere create-run kwam tot hier; opnieuw draaien hervat
                print(f"  {name}: al gekoppeld aan {repo}@{br}")
            else:
                connect(ws, name, kind)
        if "INTEGRATION_DATA" in kinds:
            register_integration_data(find_workspace(dev_workspace_name("INTEGRATION_DATA")))

    elif mode == "new_feature":
        require_clean(kinds)
        for kind in kinds:
            name = dev_workspace_name(kind)
            ws = find_workspace(name)
            if kind in DOMAIN_TYPES and domain_of(ws) != re.sub(r"_[A-Z]+$", "", repo_name(kind)).lower():
                raise RuntimeError(f"{name} hoort bij een ander domein ({domain_of(ws)}): gebruik switch_domain")
            connect(ws, name, kind)

    elif mode == "switch_domain":
        if not confirm_empty:
            raise RuntimeError("switch_domain gooit je domein-workspaces leeg (ook je eigen Gold-data). "
                               "Zet confirm_empty = True om te bevestigen.")
        require_clean(kinds)
        for kind in reversed([k for k in DOMAIN_TYPES if k in kinds]):  # eerst wat verwijst
            empty_workspace(find_workspace(dev_workspace_name(kind)), dev_workspace_name(kind))
        for kind in kinds:  # DATA, CODE, SEMANTIC, REPORTING; INTEGRATION alleen een nieuwe branch
            connect(find_workspace(dev_workspace_name(kind)), dev_workspace_name(kind), kind)

    elif mode != "bind":
        raise ValueError("mode: create, new_feature, switch_domain of bind")

    bind()


# Uitvoeren

In [ ]:
run()